In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, sys
os.chdir("/content/drive/MyDrive/Cape Stone Project 2026")
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())


In [ ]:
# Phase 2: Data Cleaning and Transformation

import pandas as pd
from src.utils import load_csv

# Load raw data
customers = load_csv("data/raw/customers.csv")
orders = load_csv("data/raw/orders.csv")
order_items = load_csv("data/raw/order_items.csv")
products = load_csv("data/raw/products.csv")
support_tickets = load_csv("data/raw/support_tickets.csv")

# -----------------------------
# Cleaning Customers
# -----------------------------
customers["date_of_birth"] = pd.to_datetime(customers["date_of_birth"], errors="coerce")
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
customers["membership_tier"] = customers["membership_tier"].str.title()
customers = customers.drop_duplicates(subset="customer_id")

# -----------------------------
# Cleaning Orders
# -----------------------------
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders["total_amount"] = pd.to_numeric(orders["total_amount"], errors="coerce")
orders["order_status"] = orders["order_status"].str.title()
orders = orders.drop_duplicates(subset="order_id")

# -----------------------------
# Cleaning Order Items
# -----------------------------
order_items["quantity"] = pd.to_numeric(order_items["quantity"], errors="coerce").fillna(order_items["quantity"].median())
order_items = order_items.drop_duplicates(subset=["order_id","product_id"])

# -----------------------------
# Cleaning Products
# -----------------------------
products["unit_cost"] = pd.to_numeric(products["unit_cost"], errors="coerce").fillna(products["unit_cost"].median())
products["unit_price"] = pd.to_numeric(products["unit_price"], errors="coerce").fillna(products["unit_price"].median())
products["launch_date"] = pd.to_datetime(products["launch_date"], errors="coerce")
products = products.drop_duplicates(subset="product_id")

# -----------------------------
# Cleaning Support Tickets
# -----------------------------
support_tickets["ticket_date"] = pd.to_datetime(support_tickets["ticket_date"], errors="coerce")
support_tickets["resolution_time_hours"] = pd.to_numeric(support_tickets["resolution_time_hours"], errors="coerce")
support_tickets["issue_category"] = support_tickets["issue_category"].str.title()
support_tickets = support_tickets.drop_duplicates(subset="ticket_id")

# -----------------------------
# Save Cleaned Data
# -----------------------------
customers.to_csv("data/processed/customers_clean.csv", index=False)
orders.to_csv("data/processed/orders_clean.csv", index=False)
order_items.to_csv("data/processed/order_items_clean.csv", index=False)
products.to_csv("data/processed/products_clean.csv", index=False)
support_tickets.to_csv("data/processed/support_tickets_clean.csv", index=False)


In [ ]:
# Phase 7: Exploratory Data Analysis — Churn Analysis

import pandas as pd

# Load cleaned customers and orders
customers = pd.read_csv("/content/drive/MyDrive/Cape Stone Project 2026/data/processed/customers_clean.csv")
orders = pd.read_csv("/content/drive/MyDrive/Cape Stone Project 2026/data/processed/orders_clean.csv")

# Step 1: Define churn cutoff
cutoff = pd.Timestamp("2026-09-24") - pd.Timedelta(days=365)

# Last order per customer
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
last_order = orders.groupby("customer_id")["order_date"].max()

# Churn flag: 1 = churned, 0 = active
churn_flag = (last_order < cutoff).astype(int)

# Overall churn rate
overall_churn_rate = churn_flag.mean()

# Merge with customers for segmentation
customer_churn = customers.merge(churn_flag.rename("churn_flag"), on="customer_id", how="left")

# Step 2: Churn by segments
tier_churn = customer_churn.groupby("membership_tier")["churn_flag"].mean()
channel_churn = customer_churn.groupby("preferred_channel")["churn_flag"].mean()
state_churn = customer_churn.groupby("state")["churn_flag"].mean()
referral_churn = customer_churn.groupby("referral_source")["churn_flag"].mean()

# ✅ FIX: Convert signup_date and create signup_year column
customer_churn["signup_date"] = pd.to_datetime(customer_churn["signup_date"], errors="coerce")
customer_churn["signup_year"] = customer_churn["signup_date"].dt.year

# Drop rows with missing signup_year
customer_churn = customer_churn.dropna(subset=["signup_year"])

# Now group by signup_year
cohort_churn = customer_churn.groupby("signup_year")["churn_flag"].mean()

# Step 3: Print results
print(f"Overall Churn Rate: {overall_churn_rate:.2%}")
print("\nChurn by Tier:")
print(tier_churn)
print("\nChurn by Channel:")
print(channel_churn)
print("\nChurn by State:")
print(state_churn)
print("\nChurn by Referral Source:")
print(referral_churn)
print("\nChurn by Signup Cohort:")
print(cohort_churn)


Overall Churn Rate: 50.25%

Churn by Tier:
membership_tier
Bronze      0.498810
Gold        0.507433
Platinum    0.497283
Silver      0.506234
Name: churn_flag, dtype: float64

Churn by Channel:
preferred_channel
In-Store      0.513425
Mobile App    0.502677
Online        0.491160
Name: churn_flag, dtype: float64

Churn by State:
state
ARIZONA           0.481100
CALIFORNIA        0.506757
FLORIDA           0.489437
GEORGIA           0.511945
ILLINOIS          0.503497
IN                0.459016
INDIANA           0.464286
MARYLAND          0.482625
MASSACHUSETTS     0.492857
MICHIGAN          0.500000
MISSOURI          0.566038
MISSSOURI         0.459259
NEW JERSEY        0.527344
NEW YORK          0.461279
NORTH CAROLINA    0.528926
OHIO              0.509579
PENNSYLVANIA      0.507194
TENNESSEE         0.522059
TEXAS             0.527881
VARGINIA          0.507353
VIRGINIA          0.476923
WASHINGTON        0.503448
WASHINGTON DC     0.451128
WISCONSIN         0.551971
Name: churn_fl

In [ ]:
print(customer_churn.dtypes)
print(customer_churn["signup_date"].head())


customer_id                  object
first_name                   object
last_name                    object
email                        object
phone                        object
gender                       object
date_of_birth                object
signup_date          datetime64[ns]
state                        object
city                         object
membership_tier              object
preferred_channel            object
marketing_opt_in             object
referral_source              object
churn_flag                  float64
signup_year                   int32
dtype: object
0   2026-05-28
1   2022-12-23
2   2022-07-10
3   2022-11-03
4   2023-07-05
Name: signup_date, dtype: datetime64[ns]
